# Stage TabFormer data for the fine-tuning API

This notebook converts the TabFormer Delta table into deterministic train and evaluation files for a `CHAT_COMPLETION` run. Each line is one JSON object containing a `messages` array whose roles alternate `system`, `user`, and `assistant`, as required by the [fine-tuning data format](https://docs.databricks.com/aws/en/large-language-models/foundation-model-training/data-preparation#prepare-data-for-chat-completion).

In [ ]:
%pip install pyyaml

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pathlib import Path
import json

import yaml
from pyspark.sql import functions as F

CONFIG_FILE = "config.yaml"


def resolve_config_path() -> Path:
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent / CONFIG_FILE)
    candidates.append(Path.cwd() / CONFIG_FILE)

    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
    except Exception:
        pass
    else:
        candidates.append(
            Path("/Workspace")
            / notebook_path.lstrip("/").rsplit("/", 1)[0]
            / CONFIG_FILE
        )

    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate
    searched = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find {CONFIG_FILE}; searched: {searched}")


config_path = resolve_config_path()
with config_path.open("r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

if not isinstance(config, dict):
    raise ValueError(f"Expected a YAML mapping in {config_path}")


def required_mapping(mapping: dict, key: str) -> dict:
    value = mapping.get(key)
    if not isinstance(value, dict):
        raise ValueError(f"Config value {key} must be a mapping")
    return value


def required_string(mapping: dict, key: str) -> str:
    value = str(mapping.get(key, "")).strip()
    if not value:
        raise ValueError(f"Missing required config value: {key}")
    return value


def quote_identifier(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"


data_config = required_mapping(config, "data")
catalog = required_string(data_config, "catalog")
schema = required_string(data_config, "schema")
table = required_string(data_config, "table")
volume = required_string(data_config, "volume")
output_dir = required_string(data_config, "output_dir").strip("/")
train_file = required_string(data_config, "train_file")
eval_file = required_string(data_config, "eval_file")
system_prompt = required_string(data_config, "system_prompt")
examples_per_label = int(data_config.get("examples_per_label", 0))
eval_fraction = float(data_config.get("eval_fraction", 0))
split_seed = int(data_config.get("split_seed", 0))

if examples_per_label <= 0:
    raise ValueError("examples_per_label must be greater than zero")
if not 0 < eval_fraction < 1:
    raise ValueError("eval_fraction must be between zero and one")
if not train_file.endswith(".jsonl") or not eval_file.endswith(".jsonl"):
    raise ValueError("train_file and eval_file must use the .jsonl extension")

source_table = ".".join(map(quote_identifier, (catalog, schema, table)))
volume_name = ".".join(map(quote_identifier, (catalog, schema, volume)))
output_path = f"dbfs:/Volumes/{catalog}/{schema}/{volume}/{output_dir}"
train_path = f"{output_path}/{train_file}"
eval_path = f"{output_path}/{eval_file}"
print(f"Using configuration from {config_path}")

## Select a balanced example dataset

In [ ]:
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {quote_identifier(catalog)}."
    f"{quote_identifier(schema)}"
)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

transactions = spark.table(source_table)
required_columns = {
    "user_id",
    "card_id",
    "year",
    "month",
    "day",
    "time",
    "amount",
    "use_chip",
    "merchant_name",
    "merchant_city",
    "merchant_state",
    "zip_code",
    "mcc",
    "errors",
    "is_fraud",
}
missing_columns = sorted(required_columns.difference(transactions.columns))
if missing_columns:
    raise ValueError(f"Missing columns in {source_table}: {missing_columns}")

fraud_text = F.lower(F.trim(F.col("is_fraud").cast("string")))
labeled_transactions = (
    transactions.withColumn(
        "fraud_label",
        F.when(fraud_text.isin("yes", "true", "1"), F.lit("fraud"))
        .when(
            fraud_text.isin("no", "false", "0"),
            F.lit("non_fraud"),
        ),
    )
    .filter(F.col("fraud_label").isNotNull())
    .withColumn(
        "_sample_hash",
        F.xxhash64(
            F.col("user_id"),
            F.col("card_id"),
            F.col("year"),
            F.col("month"),
            F.col("day"),
            F.col("time"),
            F.col("merchant_name"),
        ),
    )
)

fraud_examples = (
    labeled_transactions.filter(F.col("fraud_label") == "fraud")
    .orderBy("_sample_hash")
    .limit(examples_per_label)
)
non_fraud_examples = (
    labeled_transactions.filter(F.col("fraud_label") == "non_fraud")
    .orderBy("_sample_hash")
    .limit(examples_per_label)
)
examples = fraud_examples.unionByName(non_fraud_examples)

label_counts = {
    row["fraud_label"]: row["count"]
    for row in examples.groupBy("fraud_label").count().collect()
}
expected_label_counts = {
    "fraud": examples_per_label,
    "non_fraud": examples_per_label,
}
if label_counts != expected_label_counts:
    raise ValueError(
        f"Expected {expected_label_counts}, found {label_counts}"
    )
print(f"Selected examples by label: {label_counts}")

## Format each transaction as a chat session

In [ ]:
def text_value(column_name: str):
    value = F.trim(F.col(column_name).cast("string"))
    return F.when(value.isNull() | (F.length(value) == 0), "unknown").otherwise(
        value
    )


user_message = F.concat(
    F.lit("Classify this transaction as fraud or non_fraud.\n"),
    F.lit("User ID: "),
    text_value("user_id"),
    F.lit("\nCard ID: "),
    text_value("card_id"),
    F.lit("\nDate: "),
    text_value("year"),
    F.lit("-"),
    text_value("month"),
    F.lit("-"),
    text_value("day"),
    F.lit("\nTime: "),
    text_value("time"),
    F.lit("\nAmount: "),
    text_value("amount"),
    F.lit("\nPayment method: "),
    text_value("use_chip"),
    F.lit("\nMerchant ID: "),
    text_value("merchant_name"),
    F.lit("\nMerchant city: "),
    text_value("merchant_city"),
    F.lit("\nMerchant state: "),
    text_value("merchant_state"),
    F.lit("\nZIP code: "),
    text_value("zip_code"),
    F.lit("\nMCC: "),
    text_value("mcc"),
    F.lit("\nErrors: "),
    text_value("errors"),
)

messages = F.array(
    F.struct(
        F.lit("system").alias("role"),
        F.lit(system_prompt).alias("content"),
    ),
    F.struct(
        F.lit("user").alias("role"),
        user_message.alias("content"),
    ),
    F.struct(
        F.lit("assistant").alias("role"),
        F.col("fraud_label").alias("content"),
    ),
)

eval_threshold = round(eval_fraction * 1_000_000)
chat_examples = (
    examples.withColumn("messages", messages)
    .withColumn(
        "_split_bucket",
        F.pmod(
            F.xxhash64(F.lit(split_seed), F.col("_sample_hash")),
            F.lit(1_000_000),
        ),
    )
    .withColumn(
        "split",
        F.when(F.col("_split_bucket") < eval_threshold, "eval").otherwise(
            "train"
        ),
    )
    .select("split", "fraud_label", "_sample_hash", "messages")
)

split_counts = {
    row["split"]: row["count"]
    for row in chat_examples.groupBy("split").count().collect()
}
if set(split_counts) != {"train", "eval"}:
    raise ValueError(f"Expected train and eval records, found {split_counts}")
print(f"Examples by split: {split_counts}")

## Write one JSONL file per split

In [ ]:
def write_jsonl(dataframe, destination: str) -> None:
    temporary_dir = f"{destination}.tmp"
    dbutils.fs.rm(temporary_dir, recurse=True)
    dbutils.fs.rm(destination, recurse=True)

    lines = dataframe.orderBy("_sample_hash").select(
        F.to_json(F.struct("messages")).alias("value")
    )
    lines.coalesce(1).write.mode("overwrite").text(temporary_dir)

    part_files = [
        file.path
        for file in dbutils.fs.ls(temporary_dir)
        if file.name.startswith("part-")
    ]
    if len(part_files) != 1:
        raise RuntimeError(
            f"Expected one JSONL part file in {temporary_dir}, found {part_files}"
        )

    if not dbutils.fs.mv(part_files[0], destination):
        raise RuntimeError(f"Could not move JSONL output to {destination}")
    dbutils.fs.rm(temporary_dir, recurse=True)


dbutils.fs.mkdirs(output_path)
write_jsonl(chat_examples.filter(F.col("split") == "train"), train_path)
write_jsonl(chat_examples.filter(F.col("split") == "eval"), eval_path)

def read_and_validate_first_record(path: str) -> dict:
    record = json.loads(dbutils.fs.head(path, 4096).splitlines()[0])
    messages = record.get("messages", [])
    roles = [message.get("role") for message in messages]
    if roles != ["system", "user", "assistant"]:
        raise ValueError(f"Unexpected roles in {path}: {roles}")
    if not all(isinstance(message.get("content"), str) for message in messages):
        raise ValueError(f"Every message in {path} must have string content")
    return record


first_record = read_and_validate_first_record(train_path)
read_and_validate_first_record(eval_path)

print(f"Training data: {train_path}")
print(f"Evaluation data: {eval_path}")
print(json.dumps(first_record, indent=2))